In [1]:
import pandas as pd
df = pd.read_csv("titanic.csv")

people = df[["PassengerId", "Name", "Sex", "Age", "Pclass"]]
travel = df[["PassengerId", "Ticket", "Fare", "Cabin", "Embarked"]]

In [2]:
pd.merge(people, travel, on="PassengerId")

,PassengerId,Name,Sex,Age,Pclass,Ticket,Fare,Cabin,Embarked
0,1,"Braund, Mr. Owen Harris",male,22.0,3,A/5 21171,7.2500,NaN,S
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,PC 17599,71.2833,C85,C
2,3,"Heikkinen, Miss. Laina",female,26.0,3,STON/O2. 3101282,7.9250,NaN,S
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,113803,53.1000,C123,S
4,5,"Allen, Mr. William Henry",male,35.0,3,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...
886,887,"Montvila, Rev. Juozas",male,27.0,2,211536,13.0000,NaN,S
887,888,"Graham, Miss. Margaret Edith",female,19.0,1,112053,30.0000,B42,S
888,889,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,3,W./C. 6607,23.4500,NaN,S
889,890,"Behr, Mr. Karl Howell",male,26.0,1,111369,30.0000,C148,C


In [3]:
# # Predictions
# # merged.shape after joining the two full tables — how many rows, how many columns?
# Rows will be the field name provided insde the vaiable and columns will be same as that of the original titanic table
# # travel_partial has 500 rows, people has 891. What row count comes back from the inner join? From the left join?
# inner join will give 500 beacuase(only rows present in both and in left join rows from people table

# # After the left join, Fare will have nulls for the unmatched rows. How many?
# quick guess 391 maybe


In [4]:
merged = pd.merge(people, travel, on="PassengerId")
merged.shape

# make them mismatch on purpose



(891, 9)

In [5]:
travel_partial = travel.head(500)
pd.merge(people, travel_partial, on="PassengerId").shape                  # inner


(500, 9)

In [6]:
pd.merge(people, travel_partial, on="PassengerId", how="left").shape      # left


(891, 9)

In [7]:
pd.merge(people, travel_partial, on="PassengerId", how="outer").shape     # outer

(891, 9)

In [8]:
class_names = pd.DataFrame({"Pclass": [1, 2, 3], "ClassName": ["First", "Second", "Third"]})
class_names
df_named = pd.merge(df, class_names, on="Pclass", how="left")
df_named.shape
df_named[["Name", "Pclass", "ClassName"]].head()
     

,Name,Pclass,ClassName
0,"Braund, Mr. Owen Harris",3,Third
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,First
2,"Heikkinen, Miss. Laina",3,Third
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,First
4,"Allen, Mr. William Henry",3,Third


In [9]:
# Build a summary table of survival rate by class using groupby, then merge it back onto df so each passenger carries their class's overall survival rate as a column. (Hint: .reset_index() turns a groupby result into a mergeable DataFrame.)
survival_by_class = (
    df.groupby("Pclass")["Survived"]
      .mean()
      .reset_index(name="ClassSurvivalRate")
)

df = pd.merge(df, survival_by_class, on="Pclass", how="left")

print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  ClassSurvivalRate  
0      0         A/5 21171   7.2500   NaN        S           0.242363  
1      0          PC 17599  71.2833   C85        C           0.629630  
2      0  STON/O2. 3101282   7.9250   NaN        S           0.242363  
3      0            113803  53.1

In [10]:
# Merge people with a 300-row slice of travel using all four how values. Report the shape of each and explain in one line why they differ.?
travel_300 = travel.iloc[:300]
print("Inner:", pd.merge(people, travel_300, on="PassengerId", how="inner").shape)
print("Left :", pd.merge(people, travel_300, on="PassengerId", how="left").shape)
print("Right:", pd.merge(people, travel_300, on="PassengerId", how="right").shape)
print("Outer:", pd.merge(people, travel_300, on="PassengerId", how="outer").shape)
# Inner: Keeps only PassengerIds that exist in both tables, so the result has 300 rows.
# Left: Keeps all 891 passengers from people; rows not in travel_300 get NaN in the travel columns.
# Right: Keeps all 300 rows from travel_300; matching passenger details are added.
# Outer: Keeps every unique PassengerId from both tables. Since travel_300 is a subset of people, the result is 891 rows, the same as the left merge.

Inner: (300, 9)
Left : (891, 9)
Right: (300, 9)
Outer: (891, 9)


In [11]:
# Merge df with itself on PassengerId
merged = pd.merge(df, df, on="PassengerId")
print(merged.columns)
# _x = column from the left DataFrame, _y = same column from the right DataFrame.

Index(['PassengerId', 'Survived_x', 'Pclass_x', 'Name_x', 'Sex_x', 'Age_x',
       'SibSp_x', 'Parch_x', 'Ticket_x', 'Fare_x', 'Cabin_x', 'Embarked_x',
       'ClassSurvivalRate_x', 'Survived_y', 'Pclass_y', 'Name_y', 'Sex_y',
       'Age_y', 'SibSp_y', 'Parch_y', 'Ticket_y', 'Fare_y', 'Cabin_y',
       'Embarked_y', 'ClassSurvivalRate_y'],
      dtype='str')


In [12]:
# Ticket numbers repeat — several passengers share one. Count how many passengers are on the most common ticket, then explain what would happen if you merged two tables on Ticket instead of PassengerId.?
print(df["Ticket"].is_unique)

ticket_counts = df["Ticket"].value_counts()
print(ticket_counts.max())
# 49

False
7
